# SaaS Metrics Review

Loads the Gold layer (`saas_metrics_daily`, `product_engagement_daily`) and plots:

1. MRR / ARR over time
2. New / Expansion / Contraction / Churned MRR waterfall for a chosen day
3. DAU / MAU over time

Run the pipeline for a few weeks of history first (see the README's Quick Start), then run this notebook top to bottom.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

GOLD_DIR = Path("..") / "include" / "data" / "gold"

saas_metrics = pd.read_parquet(GOLD_DIR / "saas_metrics_daily")
saas_metrics["dt"] = pd.to_datetime(saas_metrics["dt"])
saas_metrics = saas_metrics.sort_values("dt").reset_index(drop=True)

engagement = pd.read_parquet(GOLD_DIR / "product_engagement_daily")
engagement["dt"] = pd.to_datetime(engagement["dt"])

saas_metrics.tail()

## MRR and ARR over time

In [ ]:
fig, ax1 = plt.subplots(figsize=(11, 4))

ax1.plot(saas_metrics["dt"], saas_metrics["mrr_total"], color="tab:blue", label="MRR")
ax1.set_ylabel("MRR (USD)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(saas_metrics["dt"], saas_metrics["arr_total"], color="tab:green", linestyle="--", label="ARR")
ax2.set_ylabel("ARR (USD)", color="tab:green")
ax2.tick_params(axis="y", labelcolor="tab:green")

fig.suptitle("MRR / ARR over time")
fig.autofmt_xdate()
plt.show()

## New / Expansion / Contraction / Churned MRR waterfall

Waterfall for the most recent day with a full start-of-period baseline (i.e. not the very first day of history, which has no D-1 snapshot to compare against — see `compute_saas_metrics_daily`).

In [ ]:
with_baseline = saas_metrics.dropna(subset=["nrr"])
row = with_baseline.iloc[-1]

mrr_start = row["mrr_total"] - row["new_mrr"] - row["expansion_mrr"] + row["contraction_mrr"] + row["churned_mrr"]

labels = ["Start", "New", "Expansion", "Contraction", "Churned", "End"]
values = [mrr_start, row["new_mrr"], row["expansion_mrr"], -row["contraction_mrr"], -row["churned_mrr"], 0]

# Running totals for a simple waterfall: each bar's bottom is the previous cumulative total.
cumulative = [values[0]]
for v in values[1:-1]:
    cumulative.append(cumulative[-1] + v)
cumulative.append(0)
values[-1] = cumulative[-2]

bottoms = [0] + cumulative[:-2] + [0]
colors = ["tab:gray", "tab:green", "tab:green", "tab:red", "tab:red", "tab:blue"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(labels, values, bottom=bottoms, color=colors)
ax.set_title(f"MRR waterfall — {row['dt'].date()}")
ax.set_ylabel("USD")
for i, (b, v) in enumerate(zip(bottoms, values, strict=True)):
    ax.text(i, b + v / 2, f"{v:,.0f}", ha="center", va="center", fontsize=8)
plt.show()

## Churn / NRR over time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(with_baseline["dt"], with_baseline["nrr"] * 100, label="NRR %")
ax.plot(with_baseline["dt"], with_baseline["customer_churn_rate"] * 100, label="Customer churn rate %")
ax.axhline(100, color="gray", linestyle=":", linewidth=1)
ax.set_ylabel("%")
ax.set_title("NRR and customer churn rate over time")
ax.legend()
fig.autofmt_xdate()
plt.show()

## DAU / MAU over time

`product_engagement_daily` is one row per customer per day; aggregate to a daily total.

In [ ]:
daily_engagement = engagement.groupby("dt")[["dau", "mau"]].sum().reset_index()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily_engagement["dt"], daily_engagement["dau"], label="DAU")
ax.plot(daily_engagement["dt"], daily_engagement["mau"], label="MAU")
ax.set_ylabel("Active accounts")
ax.set_title("DAU / MAU over time")
ax.legend()
fig.autofmt_xdate()
plt.show()

## Accounts currently flagged at risk

Usage decline + `past_due` on the most recent day (see `include/saas/metrics.py::compute_product_engagement_daily`).

In [ ]:
latest_dt = engagement["dt"].max()
at_risk_today = engagement[(engagement["dt"] == latest_dt) & (engagement["at_risk"])]
print(f"{len(at_risk_today)} account(s) flagged at risk on {latest_dt.date()}")
at_risk_today[["customer_id", "dau", "wau", "mau"]]